# Partitioned QAOA for MaxCut using `divi`

> 🚀 **Skip the local bottleneck.** Qoro is giving away **$100 in free cloud compute credits.**
> Get your API key at **[dash.qoroquantum.net](https://dash.qoroquantum.net)** to run this at scale.

## Why Cloud?

A 50-node MaxCut graph needs a **2⁵⁰ statevector** — that's over **8 petabytes of RAM.** Even with clever partitioning, running 10+ QAOA sub-circuits sequentially on your laptop means waiting for each one to finish before the next starts. Qoro's Maestro runs every partition **in parallel**, collapsing hours into minutes.

## Step 0 — Install & Authenticate

```bash
pip install qoro-divi networkx matplotlib
```

In [ ]:
import time

# Get your API key (get one free at https://dash.qoroquantum.net)
# Add your API key to a .env

import networkx as nx

from utils import generate_clustered_graph, show_graph, analyze_results

from divi.qprog.algorithms import GraphProblem
from divi.qprog.optimizers import MonteCarloOptimizer
from divi.qprog import GraphPartitioningQAOA, PartitioningConfig
from divi.backends import QoroService, ParallelSimulator, JobConfig

---

## Phase 1 — Local Toy Problem (8 nodes)

Small graph (8 nodes, 2 clusters) to prove the algorithm works. This runs comfortably on any local machine.

In [ ]:
n_qubits_local = 8
n_clusters_local = 2
inter_edges = 3
p_intra = 0.3
seed = 42

G_local, node_to_cluster_local, clusters_local = generate_clustered_graph(
    n_qubits=n_qubits_local,
    n_clusters=n_clusters_local,
    inter_edges=inter_edges,
    p_intra=p_intra,
    seed=seed,
    weight=1.0,
)

show_graph(G_local, node_to_cluster_local, n_qubits_local, n_clusters_local, inter_edges)

In [ ]:
# Classical baseline
classical_cut_size, classical_partition = nx.approximation.one_exchange(G_local, seed=1)
print(f"Classical MaxCut size: {classical_cut_size}")

In [ ]:
optim = MonteCarloOptimizer(population_size=50, n_best_sets=5)
partition_config = PartitioningConfig(
    minimum_n_clusters=2, partitioning_algorithm="spectral"
)

local_backend = ParallelSimulator()

print(f"💻 Running QAOA locally on {n_qubits_local}-node graph...")
t0 = time.time()

qaoa_local = GraphPartitioningQAOA(
    graph=G_local,
    graph_problem=GraphProblem.MAXCUT,
    n_layers=2,
    optimizer=optim,
    partitioning_config=partition_config,
    max_iterations=5,
    backend=local_backend,
    grouping_strategy="qwc",
)
qaoa_local.create_programs()
qaoa_local.run(blocking=True)
qaoa_local.aggregate_results()

local_time = time.time() - t0
print(f"✅ Phase 1 complete in {local_time:.1f}s")

In [ ]:
analyze_results(G_local, qaoa_local.solution, classical_cut_size, use_index=False)

---

## Phase 2 — Scale Up with QoroService (50 nodes)

50-node graph → 10+ partitions. Each partition is dispatched to Qoro Maestro for parallel execution. This would take hours locally.

**Requirements:**
Create a `.env` file in the repo root:
```
QORO_API_KEY="your_api_key_here"
```
👉 **[Get your free API key →](https://dash.qoroquantum.net)**

In [ ]:
n_qubits_cloud = 50
n_clusters_cloud = 5

G_cloud, node_to_cluster_cloud, clusters_cloud = generate_clustered_graph(
    n_qubits=n_qubits_cloud, n_clusters=n_clusters_cloud,
    inter_edges=5, p_intra=0.2, seed=seed, weight=1.0,
)
show_graph(G_cloud, node_to_cluster_cloud, n_qubits_cloud, n_clusters_cloud, 5)

classical_cut_size_cloud, _ = nx.approximation.one_exchange(G_cloud, seed=1)

partition_config_cloud = PartitioningConfig(
    max_n_nodes_per_cluster=15, partitioning_algorithm="spectral"
)
qoro_service = QoroService(job_config=JobConfig(shots=50_000))

print(f"☁️  Routing {n_qubits_cloud}-node graph to Qoro Maestro...")
t0 = time.time()

qaoa_cloud = GraphPartitioningQAOA(
    graph=G_cloud, graph_problem=GraphProblem.MAXCUT,
    n_layers=2, optimizer=optim,
    partitioning_config=partition_config_cloud,
    max_iterations=5, backend=qoro_service, grouping_strategy="qwc",
)
qaoa_cloud.create_programs()
qaoa_cloud.run(blocking=True)
qaoa_cloud.aggregate_results()

cloud_time = time.time() - t0
print(f"\n✅ Phase 2 complete in {cloud_time:.1f}s")
print(f"⚡ Local (Phase 1): {local_time:.1f}s for {n_qubits_local} nodes")
print(f"⚡ Cloud (Phase 2): {cloud_time:.1f}s for {n_qubits_cloud} nodes")

analyze_results(G_cloud, qaoa_cloud.solution, classical_cut_size_cloud, use_index=False)

---

## Ready to Scale?

Your laptop handled 8 nodes. Qoro Maestro handled 50. Same code, cloud scale.

👉 **[Get your free API key at dash.qoroquantum.net](https://dash.qoroquantum.net)** — $100 in credits, no credit card required.